# Simulating some NR events

Pueh Leng Tan, 10 March 2026

In [1]:
import os
from time import time

import numpy as np
import pandas as pd
import scipy.stats as sps
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import jax.numpy as jnp
import multihist as mh

import appletree as apt
from appletree.utils import get_file_path

import aptext

XLA_PYTHON_CLIENT_PREALLOCATE is set to false
XLA_PYTHON_CLIENT_ALLOCATOR is set to platform
Using aptext package from https://github.com/XENONnT/applefiles


In [2]:
# constrain the GPU memory usage
apt.set_gpu_memory_usage(0.5)

## Define component

### ComponentSim

In [3]:
# Initialize component
#nr = apt.NR(bins=[bins_cs1, bins_cs2], bins_type="irreg")
#nr = apt.NR()
nr = apt.components.NRBand2f()
# i should write my own stuff (or use minghao's) under components of aptext of apt
# this component tells me exactly which steps to do, just like registering cuts in cutax
# https://github.com/XENONnT/applefiles/blob/cbb3139150526e647d1e2985f2079577f8218cd3/aptext/components/two_fold.py#L121-L146

In [4]:
# Deduce the workflow(datastructure)
nr.deduce(data_names=["cs1", "cs2"], func_name="simulate")  # 'eff'(efficiency) is always simulated
# above line basically says that i want cs1, cs2, go deduce and gather ingredients required to compute those

nr.rate_name = "nr_rate"  # also we have to specify a normalization factor of the component

# Compile ER script
# This is meta-programing because  appletree can generate codes dynamically
nr.compile()

RuntimeError: Can not find g4_ambe_sr0.npy, please check your file system

In [ ]:
nr.show_config()
# all the ingredients it needs
# if current = None, means it uses default

,option,default,current,applies_to,help
0,s1_bias_2f,2fold_s1_bias_sr0.json,None,[s1_area],2fold S1 reconstruction bias
1,s1_smear_2f,2fold_s1_smearing_sr0.json,None,[s1_area],2fold S1 reconstruction smearing
2,s1_correction,_s1_correction.json,None,[s1_correction],S1 xyz correction on reconstructed positions
3,s1_lce,_s1_correction.json,None,[s1_lce],S1 light collection efficiency
4,g4,"[g4_ambe_sr0.npy, 1000000, 7]","[g4_ambe_sr0.npy, 1000000, 7]","[energy, x, y, z, time, s2_tag, event_id, g4_e...",full-chain geant4 input file
5,ly,"[_nr_ly.json, _nr_ly.json, _nr_ly.json, t_ly]","[_nr_ly.json, _nr_ly.json, _nr_ly.json, t_ly]",[light_yield],Light yield curve
6,s2_bias,_s2_bias.json,None,[s2_area_naive],S2 reconstruction bias
7,s2_smear,_s2_smearing.json,None,[s2_area_naive],S2 reconstruction smearing
8,s2_bias,_s2_bias.json,None,[alt_s2_area_naive],S2 reconstruction bias
9,s2_smear,_s2_smearing.json,None,[alt_s2_area_naive],S2 reconstruction smearing


In [11]:
apt.utils.get_file_path("g4_ambe_sr0.npy")

RuntimeError: Can not find g4_ambe_sr0.npy, please check your file system

In [12]:
os.path.exists("g4_ambe_sr0.npy")

False

In [10]:
apt.utils.get_file_path??

Signature: apt.utils.get_file_path(fname)
Source:   
@export
def get_file_path(fname):
    """Find the full path to the resource file.

    Try 5 methods in the following order.
        * fname begin with '/', return absolute path
        * url_base begin with '/', return url_base + name
        * can get file from _get_abspath, return appletree internal file path
        * can be found in local installed ntauxfiles, return ntauxfiles absolute path
        * can be downloaded from MongoDB, download and return cached path

    """
    # 1. From absolute path if file exists
    # Usually Config.default is a absolute path
    if os.path.isfile(fname):
        return fname

    # 2. From local folder
    # Use url_base as prefix
    if "url_base" in _cached_configs.keys():
        url_base = _cached_configs["url_base"]

        if url_base.startswith("/"):
            fpath = os.path.join(url_base, fname)
            if os.path.exists(fpath):
                warn(f"Load {fname} successfull

In [5]:
# For reference, this is the compiled code, the function is stored in appletree.share._cached_functions
# Initialize component
print('NR')
print(nr.code)


NR
from functools import partial
from jax import jit
from appletree.plugins import BootstrapMS
from appletree.plugins import SigmaChargeYield
from appletree.plugins import MSDriftLoss
from appletree.plugins import NumberElectron
from appletree.plugins import MSElectronDrifted
from appletree.plugins import MSElectronMerge
from appletree.plugins import MSPositionRecon
from appletree.plugins import MSS2LCEAlt
from appletree.plugins import MSS2PEAlt
from appletree.plugins import MSS2CorrectionAlt
from appletree.plugins import MSS2Alt
from appletree.plugins import MSS2LCEMain
from appletree.plugins import MSS2PEMain
from appletree.plugins import MSS2CorrectionMain
from appletree.plugins import MSS2Main
from appletree.plugins import MScS2Alt
from appletree.plugins import MScS2Main
from appletree.plugins import MSSwapMainAltNaive
from appletree.plugins import MSSwappingMainAlt
from appletree.plugins import SigmaLightYield
from appletree.plugins import MSS1LCE
from appletree.plugins import Num

## Load parameters

In [ ]:
# Of course we have to load parameters (and their priors) in simulation (who the hell writes such comments..)
par_manager = apt.Parameter(get_file_path("nr_low.json")) # 13 parameters total
#par_manager = apt.Parameter(get_file_path("nestv2.json")) # 17 parameters total

## Simulation

In [ ]:
num_sigmas = 6
tail_prob = sps.norm.sf(x=num_sigmas)
suggested_max_batch = sps.norm.ppf(1-tail_prob,
                                   loc=par_manager.par_config['nr_rate']['init_mean'],
                                   scale=par_manager.par_config['nr_rate']['init_mean'])
print(suggested_max_batch) # number of NR events hardly gonna fluctuate above this

In [ ]:
batch_size = int(14e3)
#num_sims = int(3000)
num_sims = int(30)

param_bag = []
events_bag = []

for _mc in range(num_sims):
    #print(_mc)
    key = apt.randgen.get_key(seed=_mc)
    par_manager.sample_prior() # sampling from prior
    parameters = par_manager.get_all_parameter()

    key, tmp = nr.simulate(key, batch_size, parameters)
    events = np.asarray(jnp.stack(tmp, axis=1))   # shape (n, 3)
    #print(events)
    param_bag.append(parameters.copy())
    events_bag.append(events)

In [ ]:
target = 200_000

24.6/3000*target/60. # mins

In [ ]:
_mc = 2

this_params = param_bag[_mc]
this_events = events_bag[_mc][:, :2]
this_eff = events_bag[_mc][:, -1].astype(bool)

In [ ]:
this_params, len(this_params)

In [ ]:
plt.plot(this_events[:, 0], this_events[:, 1], '.', label='all events')
plt.plot(this_events[:, 0][~this_eff], this_events[:, 1][~this_eff], 'rx', label='eff = 0')

plt.yscale('log')
plt.xlabel('cs1 [PE]')
plt.ylabel('cs2 [PE]')
plt.legend()

In [ ]:
# todo:
# 0. realistic parameters file
# 1. save the simulated data
# 2. visualize the simulated data
# 3. think of a way to sample number of events according to the rate parameter